In [93]:
# Code to map Maven Lead elements to Origami Lead elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# March 2026

In [94]:
# For each Maven Lead data element, check if there is an exact or close matching data element in the origami Lead package
# The output will list the maven element alongside the origami element and other supporting attributes

In [95]:
from sentence_transformers import SentenceTransformer


In [96]:
import pandas as pd
import numpy as np

In [97]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [98]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [99]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If not space, digits, uppercase letters, lowercase letters, or underscore, ignore it
        if not (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=''
        else:
            new_name+=ch
    return new_name.strip()

In [100]:
# Clean up questions or data elements
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [101]:
#df_maven.loc[:,'Model'].value_counts(dropna=False)

In [102]:
df_maven_lead = df_maven[df_maven['Model']=='Childhood Lead']
len(df_maven_lead)

606

In [103]:
# No duplicates
df_maven_lead[df_maven_lead.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [104]:
# Read origami elements
df_origami = pd.read_csv('origami/Custom__CaseLead.csv',na_filter=False)
len(df_origami)

783

In [105]:
df_origami.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 783 entries, 0 to 782
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Field                 783 non-null    object
 1   Type                  783 non-null    object
 2   Label                 783 non-null    object
 3   Required in Database  783 non-null    bool  
 4   Can't Protect         783 non-null    bool  
 5   FK Table              783 non-null    object
 6   CodeTypeID            783 non-null    object
 7   Static Codes          783 non-null    object
 8   Field Notes           783 non-null    object
 9   Global Comments       783 non-null    object
 10  Local Comments        783 non-null    object
dtypes: bool(2), object(9)
memory usage: 56.7+ KB


In [106]:
#df_origami.loc[:,'Label'].value_counts(dropna=False)

In [107]:
# Get the two columns of interest and non-blank values
df_origami = df_origami[['Field','Label']]
df_origami = df_origami[df_origami['Label']!='']
len(df_origami)

401

In [108]:
df_origami = df_origami.reset_index(drop=True)

In [109]:
# Original label to be used later for output
original_labels = list(df_origami.Label)
#original_labels

In [110]:
# Process the column values (eliminate the values in square brackets)
df_origami.loc[:,'Label'] = [ label.split(']')[1].strip() if ']' in label else label.strip() for label in df_origami.loc[:,'Label']]

In [111]:
# Clean up column names in both data sets
df_origami.loc[:,'Label'] =[ clean_up(q) for q in df_origami.loc[:,'Label']]

In [112]:
#df_origami.loc[:,'Label'].value_counts()

In [113]:
# duplicates labels exist but they are labels to different questions, so keep them
#df_origami[df_origami.duplicated(['Label'],keep=False)]

In [114]:
# Do a semantic match between Maven Lead elements and Origami Lead labels

In [115]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven_lead.Question]
origami_elements = [str.replace(col,'_',' ') for col in df_origami.Label]

In [116]:
#origami_elements

In [117]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [118]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [119]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

401
606


array([[-0.0020506 , -0.04598626, -0.04524118, ...,  0.01019603,
         0.0600717 , -0.01581826],
       [-0.080986  ,  0.06571371, -0.00726671, ..., -0.03971622,
         0.06275451, -0.00660688],
       [-0.01527976,  0.01455189, -0.0803012 , ...,  0.00756959,
         0.01350452, -0.00275409],
       ...,
       [-0.01195142,  0.04254857,  0.08214842, ..., -0.01518289,
        -0.02453376,  0.0252025 ],
       [-0.06644832,  0.03604519,  0.10580117, ..., -0.01695742,
         0.02714118,  0.0212451 ],
       [-0.0375696 ,  0.05908274,  0.00948133, ...,  0.02978237,
        -0.01489797, -0.0133038 ]], dtype=float32)

In [120]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [121]:
similarities

tensor([[ 0.0258,  0.1144, -0.0019,  ...,  0.2399,  0.0640,  0.2023],
        [ 0.1138,  0.1608, -0.0025,  ...,  0.0504,  0.1517,  0.0842],
        [ 0.0338, -0.0344, -0.0397,  ...,  0.1266,  0.0640,  0.1591],
        ...,
        [ 0.1462,  0.2037,  0.1194,  ...,  0.1201,  0.1136,  0.1390],
        [ 0.1729,  0.1440,  0.1463,  ...,  0.0985,  0.1070,  0.1319],
        [ 0.1609,  0.0577,  0.0730,  ...,  0.0437,  0.0396,  0.0893]])

In [122]:
type(similarities)
len(similarities)

606

In [123]:
df_similarities = pd.DataFrame(similarities,index=df_maven_lead.Question,columns=original_labels)

In [124]:
#df_similarities

In [125]:
df_similarities.to_csv('output/similarity_scores_lead.csv')

In [126]:
#df_similarities.loc['DEVELOPMENTAL_SCREENING',:]

In [127]:
df_maven_lead = df_maven_lead.reset_index(drop=True)
df_maven_lead.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message', 'mmg_element', 'cste_element'],
      dtype='object')

In [128]:
df_output = df_maven_lead[['Model','depth','Question','Message','mmg_element','cste_element']]

In [129]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.92
    v = df_similarities.loc[idx,:]>0.92

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            if df_similarities.loc[idx,col] > max:
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'Question':idx,'target_label':max_col})



In [130]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

606


In [131]:
# Merge df_filtered with df_origami, first set df_origami to original labels
df_origami.Label = original_labels


In [132]:
df_merge = pd.merge(df_filtered,df_origami,left_on='target_label',right_on='Label',how='left')

In [133]:
#df_merge

In [134]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='Question')
#df_final

In [135]:
# Add, rename, remove columns as needed
df_final['target_table'] = 'Custom__CaseLead'
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [136]:
df_final = df_final[["Model",'Question','target_table','target_field','target_label','mmg_element','cste_element','Message','depth']]
#df_final

In [137]:
#df_final[df_final['target_field'].isna()]

In [138]:
df_final.to_csv('output/maven_lead_origami_similarity_match.csv',index=False)